# M0: independent Gaussian copula

This notebook studies the null-dependence reference distribution for one complete physical group $\mathcal E_g$. It does **not** change Chronos-2 quantiles. Its scientific role is to ask how the aggregate behaves if the fixed entity-wise predictive marginals are coupled independently.

By definition, for every forecast instance $i$ and lead $\tau$,

$$R_{g,\tau}^{(i)}=I_{K_g},\qquad \mathbf X\sim\mathcal N(0,I_{K_g}),\qquad U_k=\Phi(X_k).$$

Each $U_k$ is then projected to the same fixed finite Chronos marginal grid used by every other method.

In [ ]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from simcast.config import SimcastConfig, deep_merge, load_config
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config

CONFIG_FILE = 'configs/powertech2027/transformer.yaml'  # Change to any full-group YAML.
OVERRIDES = ()
TRAIN_IF_MISSING = False  # Default is read-only.
RUN_EVALUATION = False    # Default is read-only.
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'notebook_walkthrough' / 'm0_independent'

base_config = load_config(PROJECT_ROOT / CONFIG_FILE, overrides=OVERRIDES)
config = SimcastConfig.model_validate(deep_merge(base_config.model_dump(mode='python'), {'dependence': {'method': 'independent'}}))
cache_name = config.output.cache_name or f'liander2024_{config.data.entity_type}'
CACHE_DIR = PROJECT_ROOT / config.output.cache_dir / cache_name
assert CACHE_DIR.is_dir(), f'Missing cache: {CACHE_DIR}. Build it with the CLI first.'
library = load_pit_library(CACHE_DIR, access='training')
ds = library.dataset
entity_ids = [str(x) for x in ds.entity_id.values]
K_g = len(entity_ids)
assert entity_ids == config.protocol.ordered_entity_ids and K_g == config.protocol.entity_count
print(f'group={config.data.entity_type}, K_g={K_g}, cache={CACHE_DIR}')

## What is fixed, and what is being tested

The cache contains finite marginal quantiles $\hat y_{k,\tau,q_j}^{(i)}$, observed values $y_{k,\tau}^{(i)}$, and historical pseudo-scores $z_{k,\tau}^{(i)}=\Phi^{-1}(u_{k,\tau}^{(i)})$. M0 estimates no dependence parameter from $\mathbf z$. It is therefore not a claim that loads are independent; it is a controlled baseline in which residual scenario ranks are independent conditional on the same frozen information set.

A full vector is admissible only when $V_{g,\tau}^{(i)}=1$. If any entity is invalid, the entire vector is excluded rather than replacing $K_g$ by a smaller number.

In [ ]:
pit_z = np.asarray(ds['pit_z'].values)  # [origin, entity, lead]
valid = np.isfinite(pit_z).all(axis=1)  # [origin, lead], full-group criterion
R_m0 = np.eye(K_g)
print('array dimensions:', dict(ds.sizes))
print('valid complete (origin, lead) vectors:', int(valid.sum()), 'of', valid.size)
print('M0 correlation shape:', R_m0.shape)
assert R_m0.shape == (K_g, K_g) and np.allclose(R_m0, R_m0.T)
assert np.allclose(np.diag(R_m0), 1.0)

## From independent uniforms to an aggregate

For $m=1,\ldots,M$, M0 draws $\eta^{(m)}\sim\mathcal N(0,I_{K_g})$ and sets $U_k^{(m)}=\Phi(\eta_k^{(m)})$. The finite projection returns $\widetilde Y_{k,\tau}^{(i,m)}$ from the native grid, after which

$$\widetilde A_{g,\tau}^{(i,m)}=\sum_{k\in\mathcal E_g}\widetilde Y_{k,\tau}^{(i,m)}.$$

M0, M1, M2, and M3 all have the same discrete marginal law for each $\widetilde Y_k$. Differences in aggregate calibration or proper score arise only from which entity marginal outcomes occur together.

In [ ]:
run_dir = OUTPUT_DIR / 'independent'
if TRAIN_IF_MISSING and not run_dir.exists():
    run_dir = train_from_config(config, cache_dir=CACHE_DIR, output_dir=run_dir)
elif not run_dir.exists():
    print('No notebook M0 model artifact requested; set TRAIN_IF_MISSING=True to create a new, separate run.')

if RUN_EVALUATION:
    if not run_dir.exists():
        raise FileNotFoundError('Train M0 first or select an existing M0 run directory.')
    evaluate_from_config(config, methods=('independent',), method_runs={'independent': run_dir}, cache_dir=CACHE_DIR, output_dir=OUTPUT_DIR / 'evaluation')
else:
    print('Evaluation is disabled. Existing CLI evaluation files can be inspected without recomputation.')

In [ ]:
import pandas as pd
from IPython.display import display
metrics_file = OUTPUT_DIR / 'evaluation' / 'metrics_by_lead.csv'
if metrics_file.is_file():
    metrics = pd.read_csv(metrics_file)
    display(metrics.groupby('method', as_index=False).mean(numeric_only=True))
    metrics.pivot(index='lead', columns='method', values='mean_pinball').plot(title='Aggregate pinball loss by lead')
else:
    print('No notebook evaluation table yet. Set RUN_EVALUATION=True only when a new artifact is intended.')